### `PromptTemplate` vs `ChatPromptTemplate` 비교 (v1.0 기준)

| 구분 | PromptTemplate | ChatPromptTemplate |
| --- | --- | --- |
| **주 용도** | 단일 텍스트 완성 모델 | **채팅 기반 LLM** |
| **데이터 구조** | 단순 String | **List of Message Objects** |
| **에이전트 적합성** | 낮음 | **높음 (MessagesPlaceholder 필수 활용)** |
| **멀티모달 지원** | 불가 | **지원 (이미지/파일 블록 포함 가능)** |

※ LangChain v1.0에서 MessagesPlaceholder는 프롬프트 내에서 대화 기록(Chat History)이나 에이전트의 작업 단계(Scratchpad)와 같이 길이를 예측할 수 없는 메시지 목록을 위한 가변적 공간을 확보하는 데 사용됩니다.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# gemini_api_key = os.getenv("GEMINI_API_KEY")

True

In [2]:
from langchain_core.prompts import PromptTemplate

template = "{language} 할 수 있어?"

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='{language} 할 수 있어?')

In [5]:
prompt.invoke({"language": "한국말"})

StringPromptValue(text='한국말 할 수 있어?')

In [6]:
prompt = prompt.format(language="한국말")
prompt

'한국말 할 수 있어?'

In [7]:
from langchain_core.prompts import ChatPromptTemplate

template = "{language} 할 수 있어?"

prompt = ChatPromptTemplate.from_template(template)
prompt.invoke({"language": "Python"})

ChatPromptValue(messages=[HumanMessage(content='Python 할 수 있어?', additional_kwargs={}, response_metadata={})])

In [8]:
prompt = prompt.format(language="한국말")
prompt


'Human: 한국말 할 수 있어?'

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    # google_api_key=gemini_api_key
)

In [ ]:
response = llm.invoke(prompt)

'네, 한국말 할 수 있습니다. 무엇을 도와드릴까요?'

In [9]:
response.content

'네, 한국말 할 수 있습니다. 무엇을 도와드릴까요?'

In [8]:
response

AIMessage(content='네, 한국말 할 수 있습니다. 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--fdfaedec-c4ab-4c6f-b962-a22be671a7ea-0', usage_metadata={'input_tokens': 9, 'output_tokens': 38, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 24}})

### 코드 실행 결과 비교 분석

두 코드를 실행했을 때 반환되는 객체의 내부 구조는 다음과 같은 결정적인 차이가 있습니다.

| 구분 | `PromptTemplate` | `ChatPromptTemplate` |
| --- | --- | --- |
| **객체 성격** | **텍스트(String)** 중심 | **메시지(Message)** 중심 |
| **내부 메시지 리스트** | 없음 (`template` 속성만 존재) | 존재 (`messages` 리스트 포함) |
| **기본 역할(Role)** | 역할 구분이 없음 | 자동으로 **`human`** 역할 부여 |
| **최종 출력 타입** | `StringPromptValue` | `ChatPromptValue` |


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 1. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    # ("system", "너는 친절한 AI 어시스턴트야."),
    # {"role": "system", "content": "너는 프로그래밍 언어 전문가야."},
    SystemMessage(content="너는 친절한 AI 어시스턴트야."),
    
    # 대화 기록이 들어갈 자리를 예약 (변수명을 'history'로 지정)
    MessagesPlaceholder(variable_name="history"),
    
    # ("human", "{language} 할 수 있어?")
    # {"role": "human", "content": "{language} 할 수 있어?"},
    HumanMessagePromptTemplate.from_template("{language} 할 수 있어?")
])

# 2. 데이터 주입 및 실행 (invoke)
# 'history'에는 메시지 객체의 리스트가 들어가야 합니다.
chain_input = {
    "language": "Python",
    "history": [
        # ("human", "내 이름은 김일남이야. 답변할 때는 반드시 내 이름을 불러줘."),
        HumanMessage(content="내 이름은 김일남이야. 답변할 때는 반드시 내 이름을 불러줘."),
        # ("assistant", "네, 김일남님!")
        AIMessage(content="네, 김일남님!")
    ]
}

response = prompt.invoke(chain_input)

# 출력 확인
print(response)

messages=[SystemMessage(content='너는 친절한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='내 이름은 김일남이야. 답변할 때는 반드시 내 이름을 불러줘.', additional_kwargs={}, response_metadata={}), AIMessage(content='네, 김일남님!', additional_kwargs={}, response_metadata={}), HumanMessage(content='{language} 할 수 있어?', additional_kwargs={}, response_metadata={})]


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    # google_api_key=gemini_api_key
)

llm.invoke(response)

AIMessage(content='김일남님, 어떤 언어를 말씀하시는지 정확히 알려주시면 제가 도와드릴 수 있습니다. 저는 다양한 언어를 이해하고 생성하는 데 능숙해요!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--90ed54ae-8a3d-483a-9fcb-329eb2fe2bd9-0', usage_metadata={'input_tokens': 47, 'output_tokens': 498, 'total_tokens': 545, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 463}})